## Setup
Set `REPO_URL` to your GitHub repo, **or** leave it empty and attach the repo (uploaded as a Kaggle dataset) as an input.
Accelerator: **GPU T4 x2**. Internet: **On**.

In [ ]:
import os, glob, shutil, subprocess, json, time
REPO_URL = ""  # e.g. "https://github.com/<you>/small-model-alignment.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/small-model-alignment"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
    else:
        src = [p for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True) if "/outputs/" not in p]
        assert src, "Attach the repo as a Kaggle dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(src[0]), REPO, ignore=shutil.ignore_patterns("outputs"))
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!python -m pytest -q tests/

## Task 4: from-scratch DPO on **hh**
Attach **notebook 01's output** as an input. Policy on cuda:0 (fp32 weights + fp16 autocast), frozen fp16 reference on cuda:1.
Expected: ~2-2.5 h. Then **Save Version (Commit)**.

In [ ]:
hits = glob.glob("/kaggle/input/**/outputs/sft/best/config.json", recursive=True)
assert hits, "Attach the output of the earlier notebook as an input: outputs/sft/best/config.json"
SFT_BEST = os.path.dirname(hits[0])
print(SFT_BEST)

In [ ]:
!python scripts/train_dpo.py --sft_path "{SFT_BEST}" --dataset hh --output_dir outputs/dpo_hh

### Sanity check on 3 prompts

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh --output /tmp/sanity.json --limit 3 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1

In [ ]:
!python scripts/generate.py --model_path outputs/dpo_hh/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh --output outputs/generations/dpo_hh.json --device cuda:0 --max_new_tokens 384 --batch_size 16 --repetition_penalty 1.1
sft_gen = glob.glob("/kaggle/input/**/outputs/generations/sft_hh.json", recursive=True)[0]
shutil.copy(sft_gen, "outputs/generations/sft_hh.json")
!python scripts/build_pairs.py --baseline outputs/generations/sft_hh.json --candidate outputs/generations/dpo_hh.json --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl

In [ ]:
!python scripts/plot_metrics.py --dpo outputs/dpo_hh/metrics.jsonl --dpo_labels hh --out_dir outputs/plots_hh
!cd outputs && zip -qr ../dpo_hh_run_outputs.zip . -x "*.safetensors" && cd .. && ls -la outputs/*